# Catalog visualization interface

This notebook is intentionally thin. Reusable logic belongs in `qq_helpers.py`; this file only sets paths, loads data, calls plotting functions, and saves outputs.

In [ ]:
from pathlib import Path
import sys
import importlib

import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Folder where qq_helpers.py lives (this notebook lives in quickquake/)
HELPER_DIR = Path.cwd().resolve()

if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import qq_helpers as h
importlib.reload(h)

print("helpers:", h.__file__)

## User settings

In [ ]:
#VOLCANO = "aniakchak_0.1"

# Lines to edit: where your project's results are.
# Relative paths are relative to this notebook's folder (quickquake/).
DATA_ROOT = Path("../examples/pavlof_2021/data").resolve()   # the "data" folder next to your QQ_driver.py
NAMEBASE = "GAMMA"                                           # same as location_namebase in QQ_driver.py

CATALOG_CSV = DATA_ROOT / f"merged/output/{NAMEBASE}_hypodd_catalog_qcfiltered.csv"
STATIONS_JSON = DATA_ROOT / f"merged/input/{NAMEBASE}_station_list.json"
IMAGE_DIR = h.ensure_dir(DATA_ROOT / "images")              # figures are saved next to the results

START = "2021-07-29T00:00:00Z"
END = "2021-07-30T00:00:00Z"

DEPTH_MAX = 50.0
EXAGG = 1.0
VIEW_ELEV = 10.0
VIEW_AZIM = 265.0

## Column mapping and data load

In [ ]:
catc = h.CatalogCols(
    time="time",
    lat="latitude",
    lon="longitude",
    depth_km="depth_km",
    mag="magnitude",
)

stc = h.StationCols(
    station_id="station",
    lat="latitude",
    lon="longitude",
    elev_km="elev_km",
)

df_catalog = h.load_catalog_csv(CATALOG_CSV, catc=catc)
df_stations = h.load_stations_json(STATIONS_JSON)

print("Catalog shape:", df_catalog.shape)
print("Catalog columns:", df_catalog.columns.tolist())
print("Stations shape:", df_stations.shape)
print("Stations columns:", df_stations.columns.tolist())

display(df_catalog.head())
display(df_stations.head())

## Basic event-count time series

In [ ]:
fig, ax, counts = h.plot_event_count_time(
    catalog=df_catalog,
    time_col=catc.time,
    start=START,
    end=END,
    freq="D",
    figsize=(12, 5),
)

fig.savefig(IMAGE_DIR / "event_count_daily.svg", bbox_inches="tight")
plt.show()

display(counts.head())

## 2D seismicity map

In [ ]:
fig, ax = h.plot_mapview_seismicity(
    catalog=df_catalog,
    stations=df_stations,
    start=START,
    end=END,
    depth_vmin=0.0,
    depth_vmax=40.0,
    cmap_name="gist_rainbow",
    size_k=22.0,
    marker_min=6.0,
    plot_ellipse=False,
    relief_zoom=12,
    pad_lon=0.25,
    pad_lat=0.15,
    step_lon=0.5,
    step_lat=0.25,
    labelsize=20,
    figsize=(15, 8),
    savefig=False,
    dpi=600,
)

fig.savefig(IMAGE_DIR / "mapview_seismicity.svg", bbox_inches="tight")
plt.show()

## Basemap with stations only

In [ ]:
fig, ax = h.plot_mapview_basemap_stations(
    catalog=df_catalog,
    stations=df_stations,
    start=START,
    end=END,
    relief_zoom=12,
    pad_lon=0.25,
    pad_lat=0.15,
    step_lon=0.5,
    step_lat=0.25,
    labelsize=20,
    figsize=(15, 8),
    savefig=False,
    dpi=600,
)

fig.savefig(IMAGE_DIR / "mapview_stations_only.svg", bbox_inches="tight")
plt.show()

## 3D map colored by time

In [ ]:
# 3D interactive visualization: topography + seismicity

import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"   # shows the figure in VS Code and in Jupyter

df_cat_plot = df_catalog.copy()
df_cat_plot["time"] = pd.to_datetime(df_cat_plot["time"], utc=True, errors="coerce")
df_cat_plot = df_cat_plot.dropna(subset=["time"]).copy()

start_dt = pd.to_datetime(START, utc=True)
end_dt = pd.to_datetime(END, utc=True)

df_cat_plot = df_cat_plot[
    (df_cat_plot["time"] >= start_dt) &
    (df_cat_plot["time"] <= end_dt)
].copy()

fig = h.plot_catalog_3d_topo_seismicity(
    catalog=df_cat_plot,
    catc=catc,
    mode="time",
    year=None,
    dem_res="15s",
    topo_exagg=3.0,
    topo_opacity=0.98,
    colorscale="Spectral",
    show=False,
    renderer="plotly_mimetype+notebook_connected",
    save_image=None,
    image_format="svg",
    image_width=1200,
    image_height=900,
    image_engine="kaleido",
)

fig.write_html(IMAGE_DIR / "catalog_3d_interactive.html", include_plotlyjs="cdn")

fig.show()   # uses pio.renderers.default, set at the top of this cell

In [ ]:
fig, ax, meta = h.plot_mapview3d_timecolor(
    catalog=df_catalog,
    start=START,
    end=END,
    depth_max=DEPTH_MAX,
    dem_res="15s",
    exagg=EXAGG,
    topo_cmap="gist_earth_r",
    quake_cmap="magma",
    size_k=1.0,
    topo_alpha=0.35,
    topo_stride=5,
    view_elev=VIEW_ELEV,
    view_azim=VIEW_AZIM,
    n_xticks=5,
    n_yticks=5,
    tick_fontsize=13,
    cbar_tick_fontsize=18,
    figsize=(12, 9),
    savefig=False,
    dpi=800,
    catc=catc,
)

fig.savefig(IMAGE_DIR / "mapview3d_timecolor.svg", bbox_inches="tight")
plt.show()

print(meta)

## Zoomed 2D seismicity map

In [ ]:
fig, ax, meta = h.plot_mapview_seismicity_zoom(
    catalog=df_catalog,
    start=START,
    end=END,
    depth_vmin=0.0,
    depth_vmax=10.0,
    cmap_name="inferno_r",
    size_k=22.0,
    marker_min=6.0,
    plot_ellipse=False,
    relief_zoom=12,
    pad_lon=0.25,
    pad_lat=0.15,
    step_lon=0.2,
    step_lat=0.1,
    labelsize=20,
    figsize=(15, 8),
    dpi=600,
    zoom_factor=3,
    show=False,
)

fig.savefig(IMAGE_DIR / "mapview_zoom.svg", bbox_inches="tight")
plt.show()

print(meta)

## Optional timeline with spans



In [ ]:
spans = [
    ("2021-01-01", "2021-06-01", "Swarm A"),
    ("2022-03-01", "2022-05-15", "Swarm B"),
    ("2023-07-01", "2023-09-01", "Unrest"),
]

fig, ax, meta = h.plot_timeline_histogram(
    catalog=df_catalog,
    start=START,
    end=END,
    spans=spans,
    time_col=catc.time,
    fi_col="FI",  # if absent, helper uses EVENTS
    n_bins=90,
    figsize=(14, 4),
    show_legend=False,
    savefig=False,
    dpi=600,
)

fig.savefig(IMAGE_DIR / "timeline_histogram.svg", bbox_inches="tight")
plt.show()

print(meta["n_events"])

## Optional cross section

In [ ]:
# Cross-section end points (lon, lat) and half-width of the swath (km): edit for your region.
# The values below are for Aniakchak, Alaska.
PROFILE = {
    "start_lon": -158.2729,
    "start_lat": 56.87708,
    "end_lon": -157.9812,
    "end_lat": 56.87708,
    "label_start": "C'",
    "label_end": "C",
    "swath_km": 5.0,
}

In [ ]:
fig, ax = h.plot_mapview_seismicity(
    catalog=df_catalog,
    stations=df_stations,
    start=START,
    end=END,
    depth_vmin=0.0,
    depth_vmax=10.0,
    cmap_name="gist_rainbow",
    size_k=22.0,
    marker_min=6.0,
    plot_ellipse=False,
    relief_zoom=12,
    pad_lon=0.25,
    pad_lat=0.15,
    step_lon=0.5,
    step_lat=0.25,
    labelsize=20,
    figsize=(15, 8),
    savefig=False,   # important: save only after the profile line is added
    dpi=600,
)

h.add_profile_line(
    ax=ax,
    start_lon=PROFILE["start_lon"],
    start_lat=PROFILE["start_lat"],
    end_lon=PROFILE["end_lon"],
    end_lat=PROFILE["end_lat"],
    start_label=PROFILE["label_start"],
    end_label=PROFILE["label_end"],
    line_color="red",
    line_width=3.0,
    marker_size=80,
    fontsize=16,
)

fig.savefig(IMAGE_DIR / "mapview_seismicity_with_crosssection.pdf", bbox_inches="tight")
fig.savefig(IMAGE_DIR / "mapview_seismicity_with_crosssection.svg", bbox_inches="tight")
fig.savefig(IMAGE_DIR / "mapview_seismicity_with_crosssection.png", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
fig, ax, meta = h.plot_cross_section_seismicity(
    catalog=df_catalog,
    start=START,
    end=END,
    start_lon=PROFILE["start_lon"],
    start_lat=PROFILE["start_lat"],
    end_lon=PROFILE["end_lon"],
    end_lat=PROFILE["end_lat"],
    catc=catc,
    swath_km=PROFILE["swath_km"],
    depth_vmin=0.0,
    depth_vmax=DEPTH_MAX,
    depth_max_plot=DEPTH_MAX,
    cmap_name="inferno_r",
    size_k=22.0,
    marker_min=6.0,
    x_label_start=PROFILE["label_start"],
    x_label_end=PROFILE["label_end"],
    figsize=(12, 6),
)

fig.savefig(IMAGE_DIR / "cross_section_profile_C.svg", bbox_inches="tight")
plt.show()

print(meta)